# Phase 6 — Deployment

Two paths in this notebook. Run **one** based on your deployment decision in README.

| Path | When to use |
|---|---|
| **A — Online endpoint** | Real-time, one-record-at-a-time scoring via REST API |
| **B — Batch endpoint** | Async, file/folder input, large volumes |

**Cost reminder:** Online endpoints are billed per hour even when idle. Delete after testing if this is a portfolio project.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
PROJECT_NAME    = 'my-project'
MODEL_NAME      = f'{PROJECT_NAME}-model'
MODEL_VERSION   = '1'   # ← from Phase 5 output
ENDPOINT_NAME   = f'{PROJECT_NAME}-endpoint'   # must be globally unique in Azure
COMPUTE_NAME    = 'aml-cluster'   # batch only

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    BatchEndpoint,
    ModelBatchDeployment,
    ModelBatchDeploymentSettings,
    Model,
)
from azure.ai.ml.constants import AssetTypes, BatchDeploymentOutputAction
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

In [ ]:
# ── AUTH ──────────────────────────────────────────────────────────────────────
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://management.azure.com/.default')
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(f'✓ Connected to: {ml_client.workspace_name}')

---
## Path A — Online Endpoint (real-time)

Run cells in this section only. Skip to Path B if you chose batch.

In [ ]:
# ── A1: CREATE ENDPOINT ───────────────────────────────────────────────────────
online_endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    auth_mode='key',
)

ml_client.online_endpoints.begin_create_or_update(online_endpoint).result()
print(f'✓ Endpoint created: {ENDPOINT_NAME}')

In [ ]:
# ── A2: CREATE DEPLOYMENT ─────────────────────────────────────────────────────
blue_deployment = ManagedOnlineDeployment(
    name='blue',
    endpoint_name=ENDPOINT_NAME,
    model=f'azureml:{MODEL_NAME}:{MODEL_VERSION}',
    instance_type='Standard_DS3_v2',
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(blue_deployment).result()
print('✓ Deployment blue created')

In [ ]:
# ── A3: ROUTE ALL TRAFFIC TO BLUE ─────────────────────────────────────────────
online_endpoint.traffic = {'blue': 100}
ml_client.online_endpoints.begin_create_or_update(online_endpoint).result()
print('✓ Traffic: blue = 100%')

In [ ]:
# ── A4: TEST INVOKE ───────────────────────────────────────────────────────────
# Edit the sample data below to match your FEATURES list from Phase 1.
# data/sample/ has a small file you can use.
response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name='blue',
    request_file='../data/sample/sample_request.json',
)

endpoint_info = ml_client.online_endpoints.get(ENDPOINT_NAME)
print(f'✓ Prediction : {response}')
print(f'  Scoring URI: {endpoint_info.scoring_uri}')
print()
print('→ Record scoring URI in README → Endpoint section')

In [ ]:
# ── A5: CLEANUP (portfolio project — run after testing) ───────────────────────
# Uncomment and run this cell once you have confirmed the endpoint works.

# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print(f"✓ Endpoint '{ENDPOINT_NAME}' deleted — no more idle billing")

---
## Path B — Batch Endpoint (async)

Run cells in this section only. Skip Path A cells above.

In [ ]:
# ── B1: CREATE BATCH ENDPOINT ─────────────────────────────────────────────────
batch_endpoint = BatchEndpoint(
    name=ENDPOINT_NAME,
    description=f'Batch endpoint for {PROJECT_NAME}',
)

ml_client.batch_endpoints.begin_create_or_update(batch_endpoint).result()
print(f'✓ Batch endpoint created: {ENDPOINT_NAME}')

In [ ]:
# ── B2: CREATE BATCH DEPLOYMENT ───────────────────────────────────────────────
batch_deployment = ModelBatchDeployment(
    name='batch-blue',
    endpoint_name=ENDPOINT_NAME,
    model=f'azureml:{MODEL_NAME}:{MODEL_VERSION}',
    compute=COMPUTE_NAME,
    settings=ModelBatchDeploymentSettings(
        instance_count=1,
        max_concurrency_per_instance=2,
        mini_batch_size=10,
        output_action=BatchDeploymentOutputAction.APPEND_ROW,
        output_file_name='predictions.csv',
    ),
)

ml_client.batch_deployments.begin_create_or_update(batch_deployment).result()
print('✓ Batch deployment created')

In [ ]:
# ── B3: SET DEFAULT DEPLOYMENT ────────────────────────────────────────────────
batch_endpoint.defaults.deployment_name = 'batch-blue'
ml_client.batch_endpoints.begin_create_or_update(batch_endpoint).result()
print('✓ Default deployment set to batch-blue')

In [ ]:
# ── B4: INVOKE ────────────────────────────────────────────────────────────────
from azure.ai.ml import Input

batch_job = ml_client.batch_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    input=Input(
        type=AssetTypes.URI_FILE,
        path='../data/sample/batch_input.csv',
    ),
)

ml_client.jobs.stream(batch_job.name)
print(f'✓ Batch job complete: {batch_job.name}')

In [ ]:
# ── B5: DOWNLOAD PREDICTIONS ──────────────────────────────────────────────────
import os

os.makedirs('../outputs', exist_ok=True)

ml_client.jobs.download(
    name=batch_job.name,
    download_path='../outputs',
)

print('✓ Predictions downloaded → outputs/')

---
## Phase 6 Gate

- [ ] Test invoke returned expected output
- [ ] Scoring URI (online) or output path (batch) recorded in README → Endpoint section
- [ ] **If portfolio project: endpoint deleted (A5 cell uncommented and run)**

Blueprint complete.